**Author:** Emre Havazli

### Purpose

End-to-end stack processing for ALOS-2 data using ISCE2.

### Requirements

* ISCE2 installed and available as a Jupyter kernel.
* Helper scripts in the same project folder as this notebook:

  * `unpack_alos2.py`
  * `alos2_topo_wbd.py`
  * `makeALOSstack_xml.py`
* Project layout:

  ```
  project_folder/
    ├─ notebook.ipynb
    ├─ unpack_alos2.py
    ├─ alos2_topo_wbd.py
    ├─ makeALOSstack_xml.py
    └─ data/   # contains ALOS-2 .zip files (lowercase folder name)
  ```

### Workflow

1. **Unpack SLCs**
   Unzip ALOS-2 products into `dates/YYMMDD/` folders. Runs in parallel; set `workers` or it will use all CPUs.

2. **Prepare Topo (DEM & WBD)**
   Provide your `ISCE_HOME` path. Download DEM and water mask for a bounding box. Use `margin_deg` to slightly expand coverage.

3. **Create Processing XML**
   Generate the stack configuration. Five fields are required; other parameters have defaults you can adjust.

4. **Generate Commands**
   Create `cmd_X.sh` scripts (made executable) using the XML settings. Requires the path to the `alosstack` folder.

5. **(Optional) Enable Ionospheric Correction**
   Automatically uncomment the ionosphere-correction line in the relevant `cmd_X.sh` files.

6. **Run Processing**
   Execute `cmd_X.sh` scripts. Processing can be long; keep the notebook session alive or run from a terminal (e.g., with `screen`) and a simple shell wrapper to execute scripts sequentially.

---


# CONSTANTS

## **Edit these**

In [12]:
ISCE_HOME = "/home/jovyan/.local/envs/isce/lib/python3.12/site-packages/isce/"
PATH_ALOSSTACK="/home/jovyan/.local/envs/isce/share/isce2/alosStack/"
proj_folder = '/scratch/havazli/ALOS2/CA_P67_F750_A_Dec21_June22'
data_dir = proj_folder + '/data'
reference_date = "211202"
ctx = {"work_dir": proj_folder, 
       "min_lat": 36, "max_lat": 40, 
       "min_lon": -123, "max_lon": -117}

# Prepare ALOS2 zip files to be processed

In [ ]:
from unpack_alos2 import unpack_alos2
import logging
logging.basicConfig(level=logging.INFO, force=True, format="%(levelname)s: %(message)s")

In [ ]:
unpack_alos2(proj_folder, 8)

# Download DEM and water body mask

In [ ]:
import alos2_topo_wbd as topo
import logging
logging.basicConfig(level=logging.INFO, force=True, format="%(levelname)s: %(message)s")

In [ ]:
# Use minimum bounding rectangle from ZIPs + margin (default 0.05°), download DEM(1") + WBD:
res = topo.prepare_topo(ctx, use_dem=True, use_dem3=True, use_wbd=True, margin_deg=0, isce_home=ISCE_HOME)
res

# Make ALOS2 Stack xml file

## Default XML values

    number_of_subsequent_dates: int = 5
    number_of_subsequent_dates_for_estimating_ionosphere: int = 5

    interferogram_filter_strength: float = 0.5
    interferogram_filter_window_size: int = 32
    interferogram_filter_step_size: int = 4
    remove_magnitude_before_filtering: bool = True

    water_body_mask_starting_step: str = "unwrap"  # None|filt|unwrap

    do_ionospheric_phase_estimation: bool = True
    apply_ionospheric_phase_correction: bool = True

    apply_polynomial_fit_before_filtering_ionosphere_phase: bool = True
    whether_filtering_ionosphere_phase: bool = True
    apply_polynomial_fit_in_adaptive_filtering_window: bool = True
    whether_do_secondary_filtering_of_ionosphere_phase: bool = True
    maximum_window_size_for_filtering_ionosphere_phase: int = 301
    minimum_window_size_for_filtering_ionosphere_phase: int = 101
    window_size_of_secondary_filtering_of_ionosphere_phase: int = 5

    filter_subband_interferogram: bool = True
    subband_interferogram_filter_strength: float = 0.3
    subband_interferogram_filter_window_size: int = 128
    subband_interferogram_filter_step_size: int = 4
    remove_magnitude_before_filtering_subband_interferogram: bool = True

## Create XML File

In [ ]:
from make_ALOSstack_xml import generate_stackinsar_xml, write_xml

In [ ]:
coreg_dem = res['outputs']['dem_1arcsec'] # proj_folder + '/topo/demLat_N36_N40_Lon_W123_W117.dem.wgs84'
geocode_dem = res['outputs']['dem_1arcsec']
wbd = res['outputs']['wbd']

In [ ]:
xml_out = "ALOSstack.xml"

In [ ]:
xml_text = generate_stackinsar_xml(
    data_directory=data_dir,
    dem_for_coregistration=coreg_dem,
    dem_for_geocoding=geocode_dem,
    water_body=wbd,
    reference_date_of_the_stack=reference_date,
    # Optional overrides (everything below has a default):
    number_of_subsequent_dates=1,
    number_of_subsequent_dates_for_estimating_ionosphere=1,
)

In [ ]:
write_xml(xml_out, xml_text)

# Generate run files

In [ ]:
import alos2_topo_wbd as topo

In [ ]:
# cmd =[f"{PATH_ALOSSTACK}/create_cmds.py", f"-stack_par ./{xml_out}"]
cmd = ["{}create_cmds.py".format(PATH_ALOSSTACK), "-stack_par", xml_out]
print(cmd)

In [ ]:
topo.run_cmd(cmd)

In [ ]:
# Make cmd files executable
!chmod +x cmd_*.sh

# Edit cmd files to apply ionospheric correction

In [ ]:
import re

def uncomment_line(filename: str, line_number: int) -> bool:
    """
    Uncomment exactly one leading '#'(optional space) on the given 1-based line.
    Returns True if edited, False if line out of range.
    """
    with open(filename) as f:
        lines = f.readlines()
    i = line_number - 1
    if not (0 <= i < len(lines)):
        return False
    lines[i] = re.sub(r'^(\s*)#\s?', r'\1', lines[i], count=1)
    with open(filename, 'w') as f:
        f.writelines(lines)
    return True

In [ ]:
uncomment_line('cmd_3.sh', 309)

# RUN CMD FILES

In [ ]:
from pathlib import Path
proj = Path(proj_folder).expanduser().resolve()
cmds = sorted(proj.glob("cmd_*.sh"))

out = proj / "run_all.sh"
with out.open("w", encoding="utf-8") as f:
    f.write("#!/usr/bin/env bash\n")
    f.write("source /home/jovyan/omp_env.sh")
    for cmd in cmds:
        tag = cmd.stem[4:]
        f.write(f"sh {cmd.name} | tee run{tag}.log;\n")

out.chmod(out.stat().st_mode | 0o111)  # make executable
print(f"Wrote {out} with {len(cmds)} commands.")


In [ ]:
# !sh cmd_1.sh | tee run1.log;

In [ ]:
# !sh cmd_2.sh | tee run2.log;

In [ ]:
# !sh cmd_3.sh | tee run3.log;

In [ ]:
# !sh cmd_4.sh | tee run4.log;

# Make local incidence angle, layover and shadow masks

In [13]:
import get_alos_localIncAng as lia

In [14]:
los_file = f"{proj_folder}/dates_resampled/{reference_date}/insar/{reference_date}_5rlks_28alks.los"
hgt_file = f"{proj_folder}/dates_resampled/{reference_date}/insar/{reference_date}_5rlks_28alks.hgt"
lon_file = f"{proj_folder}/dates_resampled/{reference_date}/insar/{reference_date}_5rlks_28alks.lon"
lat_file = f"{proj_folder}/dates_resampled/{reference_date}/insar/{reference_date}_5rlks_28alks.lat"

In [15]:
lia_arr, shadow_arr, layover_arr = lia.run_lia(los_file, hgt_file,
                                               lon_file, lat_file,
                                               out_path=f"{proj_folder}/localIncAngle.tif",
                                               shadow_path=f"{proj_folder}/shadowMask.tif",
                                               layover_path=f"{proj_folder}/layoverMask.tif")

# Make MintPy Config File

In [16]:
config_file_content = {
    "mintpy.compute.cluster": "local",
    "mintpy.load.processor": "isce",
    "mintpy.load.metaFile": f"{proj_folder}/pairs/*-*/{reference_date}.track.xml",
    "mintpy.load.baselineDir": f"{proj_folder}/baseline",
    "mintpy.load.unwFile": f"{proj_folder}/pairs/*-*/insar/filt_*-*_5rlks_28alks.unw",
    "mintpy.load.corFile": f"{proj_folder}/pairs/*-*/insar/*-*_5rlks_28alks.cor",
    "mintpy.load.connCompFile": f"{proj_folder}/pairs/*-*/insar/filt_*-*_5rlks_28alks.unw.conncomp",
    "mintpy.load.demFile": f"{proj_folder}/dates_resampled/{reference_date}/insar/*_5rlks_28alks.hgt",
    "mintpy.load.lookupYFile": f"{proj_folder}/dates_resampled/{reference_date}/insar/*_5rlks_28alks.lat",
    "mintpy.load.lookupXFile": f"{proj_folder}/dates_resampled/{reference_date}/insar/*_5rlks_28alks.lon",
    "mintpy.load.incAngleFile": f"{proj_folder}/localIncAngle.tif",
    # "mintpy.load.shadowMaskFile:": f"{proj_folder}/shadowMask.tif",
    "mintpy.load.waterMaskFile": f"{proj_folder}/dates_resampled/{reference_date}/insar/*_5rlks_28alks.wbd",
    }

In [17]:
config_file = proj_folder + "/" + proj_folder.split('/')[-1] + ".cfg"

In [18]:
# Write config dictionary to text file
with open(config_file, "w") as f:
    f.writelines(f"{k} = {v}\n" for k, v in config_file_content.items())

# Confirm output
print(f"MintPy config file written to:\n    {config_file}\n")
with open(config_file, "r") as f:
    print(f.read())

MintPy config file written to:
    /scratch/havazli/ALOS2/CA_P67_F750_A_Dec21_June22/CA_P67_F750_A_Dec21_June22.cfg

mintpy.compute.cluster = local
mintpy.load.processor = isce
mintpy.load.metaFile = /scratch/havazli/ALOS2/CA_P67_F750_A_Dec21_June22/pairs/*-*/211202.track.xml
mintpy.load.baselineDir = /scratch/havazli/ALOS2/CA_P67_F750_A_Dec21_June22/baseline
mintpy.load.unwFile = /scratch/havazli/ALOS2/CA_P67_F750_A_Dec21_June22/pairs/*-*/insar/filt_*-*_5rlks_28alks.unw
mintpy.load.corFile = /scratch/havazli/ALOS2/CA_P67_F750_A_Dec21_June22/pairs/*-*/insar/*-*_5rlks_28alks.cor
mintpy.load.connCompFile = /scratch/havazli/ALOS2/CA_P67_F750_A_Dec21_June22/pairs/*-*/insar/filt_*-*_5rlks_28alks.unw.conncomp
mintpy.load.demFile = /scratch/havazli/ALOS2/CA_P67_F750_A_Dec21_June22/dates_resampled/211202/insar/*_5rlks_28alks.hgt
mintpy.load.lookupYFile = /scratch/havazli/ALOS2/CA_P67_F750_A_Dec21_June22/dates_resampled/211202/insar/*_5rlks_28alks.lat
mintpy.load.lookupXFile = /scratch/havazli/